# Clayton Metang — Expedition
High-level workflow using `expedition.py`. Each cell calls one stage; `x.save()` persists config after each step.

In [1]:
%load_ext autoreload
%autoreload 2
import logging
import claytonlib as clayton
from claytonlib.expedition import expedition

# --- Logging ---
# INFO shows per-write-cycle timing; DEBUG adds per-turn RNG details
logging.basicConfig(level=logging.INFO)
# logging.getLogger('claytonlib').setLevel(logging.INFO)

x = expedition("metang")
x.reload()
x.print()
x.chart_options.evaluation_frames_per_write_cycle = 5

[expedition] Reloaded from data/expeditions/metang.json
=== Expedition: metang ===
  pokemon                  metang
  key_seed                 0x0C0E02C2
  setup_delay_s            180
  max_target_s             600
  strategy                 six-bait-then-balls
  criteria                 machete-50-turns-after-5-balls
  eval_strategy            sliding_window_13
  fps_model                linear
  window                   120
  target_delay             20058
  initial_time             2000-07-24T14:45:55
  target_seeds             ['0x1C1562D2']
  metronome_histsz         10
  metronome_second_window  2
  compass_m_delay          2441
  ---
  delay_from_key           19352 frames  (322.53s)


In [16]:
x.adjust(strategy_name="six-bait-then-balls")

[expedition] strategy_name = 'six-bait-then-balls'


# Chart - Finding a target

## Create chart

In [2]:
x.precompute_chart()
x.save()

[expedition] 16:40:10  === precompute_chart ===  (2026-09-12)
[expedition] 16:40:10  charting metang key_seed=0x0C0E02C2 delay=180-600s strategy=six-bait-then-balls criteria=machete-50-turns-after-5-balls  fps_models=['linear', 'quad'] (union)  workers=12
[expedition] 16:40:28  mdmsh 1/246  elapsed 0.3m  eta ~71.8m
[expedition] 16:41:38  mdmsh 5/246  elapsed 1.5m  eta ~70.6m
[expedition] 16:43:05  mdmsh 10/246  elapsed 2.9m  eta ~68.6m
[expedition] 16:44:33  mdmsh 15/246  elapsed 4.4m  eta ~67.6m
[expedition] 16:45:58  mdmsh 20/246  elapsed 5.8m  eta ~65.5m
[expedition] 16:47:17  mdmsh 25/246  elapsed 7.1m  eta ~62.9m
[expedition] 16:48:40  mdmsh 30/246  elapsed 8.5m  eta ~61.2m
[expedition] 16:50:02  mdmsh 35/246  elapsed 9.9m  eta ~59.5m
[expedition] 16:51:02  mdmsh 40/246  elapsed 10.9m  eta ~56.0m
[expedition] 16:52:25  mdmsh 45/246  elapsed 12.3m  eta ~54.7m
[expedition] 16:54:05  mdmsh 50/246  elapsed 13.9m  eta ~54.5m
[expedition] 16:55:27  mdmsh 55/246  elapsed 15.3m  eta ~53.1

## Evaluate chart to find targets

`chart_report()` ranks the best **(boot time, commanded countdown M)** pairs across all candidate boot times (mode A), or the best M for a boot time you pass as `initial_time=` (mode B). It saves its ranked findings so `select_target()` can use them.

In [3]:
x.chart_report()
x.save()

[expedition] 17:51:13  === chart_report ===
Best (boot time, M) pairs  [top 10 of 14]  (jitter kernel, k=3.5):
   #            boot time     M (ms)  target F_b  second  P(capture)   sigma
   1  2000-08-23 14:50:34     559873       33533     565       26.5%    89.7
   2  2000-10-19 14:53:25     389583       23325     395       26.2%    74.8
   3  2000-09-22 14:58:12     221761       13265     227       26.1%    56.4
   4  2000-05-30 14:59:59     300367       17977     306       25.8%    65.7
   5  2000-06-28 14:54:46     426083       25513     431       25.6%    78.2
   6  2000-07-29 14:51:14     340404       20377     346       25.6%    69.9
   7  2000-01-01 14:00:11     502620       30101     508       25.5%    85.0
   8  2000-07-27 14:52:27     265868       15909     271       25.4%    61.8
   9  2000-06-27 14:52:54     428752       25673     434       25.2%    78.5
  10  2000-07-26 14:52:34     199607       11937     205       25.2%    53.5
[expedition] 17:51:47  best target for eac

## Choose Target

`select_target()` reads the findings `chart_report()` saved and lets you pick one. It records the chosen **boot time** (`initial_time`), **timer countdown** (`target_timer_delay` = M), and **expected battle frame** (`target_delay` = F_b) on the expedition, then saves.

In [4]:
x.select_target()

[expedition] 17:56:04  === select_target ===



Select by  [t] top ranking   [s] specific starting time   [l] reuse last (07-24 14:45:55)  (blank to cancel):  L


  -> best target for 2000-07-24 14:45:55: M=479599 ms, F_b=28721, P~25.0%
[expedition] Saved to data/expeditions/metang.json
[expedition] 17:56:05  target set: boot 2000-07-24T14:45:55, timer M=479599 ms, expected F_b=28721 (P~25.0%). Saved.
[expedition] 17:56:05  predicted battle time (m/d h:m:s): 07-24 14:54:00  (= boot + 485s; year is the chart's 2000)


{'rank': 1563,
 'initial_time': '2000-07-24T14:45:55',
 'M': 479599,
 'target_delay': 28721,
 'second': 485,
 'p': 0.24955273435550104,
 'sigma': 82.98652640650432,
 'mdmsh': [222, 14]}

## Examine target area

In [5]:
 x.check().chart_check_target_landing()


chart_check_target_landing  (jitter kernel, k=3.5)
boot=2000-07-24T14:45:55  timer M=479599 ms  ->  mean F_b=28721.0 (target_delay=28721)  sigma=83.0
RTC-second distribution (σ_S=0.52s):  485=65%  484=23%  486=12%  483=0%

=== second 485  P(S=485)=65.4%   mdmsh(m,h)=(222, 14)   battle 07-24 14:54:00
    cp(this second) = 25.35%   ->  contributes P·cp = 16.58% to the total
      frame      Δ        seed  hit    weight        w%      cumP%
    --------------------------------------------------------------
      28430   -291  0xDE0E6F0E    ✗    0.0021    0.001%     0.000%
      28431   -290  0xDE0E6F0F    ✗    0.0022    0.001%     0.000%
      28432   -289  0xDE0E6F10    ✗    0.0023    0.001%     0.000%
      28433   -288  0xDE0E6F11    ✗    0.0024    0.001%     0.000%
      28434   -287  0xDE0E6F12    ✗    0.0025    0.001%     0.000%
      28435   -286  0xDE0E6F13    ✗    0.0026    0.001%     0.000%
      28436   -285  0xDE0E6F14    ✗    0.0027    0.001%     0.000%
      28437   -284  0

{'p': 0.24955245716679053,
 'seconds': [{'second': 485,
   'p_second': 0.654282960650885,
   'cp': 0.2534802673146219},
  {'second': 484, 'p_second': 0.22652789266720144, 'cp': 0.2501930132546811},
  {'second': 486, 'p_second': 0.11540935636730296, 'cp': 0.22925888611945167},
  {'second': 483,
   'p_second': 0.003779790314610554,
   'cp': 0.15088690455302656}],
 'mismatches': None}

# Compass - Identify target

## Calibrate using metronome

In [ ]:
x.metronome_compass()
x.save()

## Finding what seed you hit in safari

`compass_safari()` builds candidates from the calibrated model: for the commanded countdown **M** (set by `select_target`) it sweeps the battle-frame window **F\* ± kσ** across second offsets **δ∈{−1,0,+1}** (off-by-one timer-start timing — each δ uses the *same* frame window). No hand-set delay window.

As you enter observed turns it ranks survivors by **posterior landing probability** (`P(land)`), shows the most-likely seed and which **δ** you hit ("timer on time / +1s late"), and flags when one candidate passes the confidence threshold. Extra commands:

- **`w`** — widen the frame (`k`) and/or second (`±K`) window and re-apply your path so far (also offered automatically on a no-match).
- The set is bounded to the seeds carrying `mass_cap` (default 0.999) of the landing probability; the Jane offload tip triggers on the *prior-weighted* effective count.

Pass `second_offsets=` / `mass_cap=` to override. Afterwards, `x.save_safari_run()` logs the identified seed, observed path, and inferred timer offset to `data/safari_runs.jsonl` (no capture required) for future model retuning.

In [ ]:
x.compass_safari()
x.save()

In [ ]:
# Loop-back: log this run (seed, observed path, inferred timer offset) for model retuning.
# No capture required — records even a fled/ambiguous run.
x.save_safari_run()

# Machete - Finding a path through seed

This is usually triggered during the "Finding what seed you hit in safari" step, but here's some manual activation anyways

## Finding a path for a single seed

In [ ]:
x.machete_one(max_turns=1000)
x.save()